# 0. Objetivo

## Etapa 1 — Leitura e União dos Arquivos Estaduais (RQUAL/Anatel)

## Objetivo

Unificar as planilhas estaduais de indicadores de qualidade (`RQUAL_8ind-*.xlsx`), obtidas no **Portal de Dados Abertos da Anatel**, em uma única base nacional consolidada.  
O objetivo é permitir análises integradas e posteriores cruzamentos com dados socioeconômicos, geográficos e demográficos.

---

## Fonte e Coleta dos Dados

Os arquivos foram extraídos manualmente do [**Portal de Dados Abertos da Anatel**](https://dados.gov.br/dataset/anatel-indicadores-de-qualidade), acessando o dataset:

> **Indicadores de Qualidade dos Serviços de Telecomunicações (RQUAL)**

Durante a coleta:
- Foram aplicados **filtros por Unidade Federativa (UF)** diretamente na interface do portal.
- Cada estado foi exportado como arquivo `.xlsx`, resultando em 12 arquivos nomeados conforme o padrão:
    - RQUAL_8ind-A.xlsx
    - RQUAL_8ind-BA.xlsx
    - RQUAL_8ind-CD.xlsx
    - RQUAL_8ind-E-G-MA.xlsx
    - RQUAL_8ind-MG.xlsx
    - RQUAL_8ind-MT-MS.xlsx
    - RQUAL_8ind-P.xlsx
    - RQUAL_8ind-RJ.xlsx
    - RQUAL_8ind-RN-RO-RR.xlsx
    - RQUAL_8ind-RS.xlsx
    - RQUAL_8ind-SC-SE-TO.xlsx
    - RQUAL_8ind-SP.xlsx

Total aproximado: **500 MB**, cobrindo todos os 27 estados brasileiros.

---

## Metodologia de Processamento

### Motivação
Foram testados diferentes fluxos de leitura e unificação de planilhas grandes:

| Estratégia | Descrição | Desempenho no Mac M4 |
|-------------|------------|----------------------|
| **XLSX → CSV → Parquet (PyArrow)** | Conversão intermediária para CSV e unificação via PyArrow | Maior tempo total (I/O elevado) |
| **XLSX → XLSX (Pandas nativo)** | Leitura direta de Excel e concatenação incremental | Mais rápido e simples |
| **PyArrow nativo** | Leitura direta via Arrow Dataset | Não suportado oficialmente para `.xlsx` |

 **Conclusão:**  
No ambiente local (Mac M4 com SSD), o fluxo direto **XLSX → XLSX** usando `pandas + openpyxl + ThreadPoolExecutor` apresentou **melhor custo-benefício**, com tempo médio de leitura **~3–5 s por arquivo** e unificação total em **menos de 1 minuto**.

---

## Arquitetura de Leitura

A rotina realiza:
1. **Detecção automática** de todos os arquivos `.xlsx` no diretório de trabalho.
2. **Leitura paralela** usando `ThreadPoolExecutor`, otimizando o uso dos núcleos M-series.
3. **Rastreamento da origem** com a coluna `__arquivo_origem`.
4. **Concatenação final** (`pd.concat`) e auditoria de consistência (linhas, colunas, duplicatas).

O código é estruturado para garantir reprodutibilidade, consistência e rastreabilidade.

---

## Resultado

- Base nacional consolidada em memória (`base`) com:
  - **>500 mil linhas**
  - **colunas padronizadas**
  - campo `__arquivo_origem` permitindo auditorias
- Tempo total de processamento: **~50 s**
- Exportação final: `base_unificada.xlsx` (ou `.parquet`, conforme necessidade)

---

## Considerações finais

- O formato `.xlsx` é suficiente nesta etapa, pois os datasets da Anatel têm estrutura tabular estável e colunas homogêneas.
- A conversão para `.parquet` é recomendada apenas **após o pré-processamento**, caso o volume cresça (ex.: >1 M linhas ou uso em Spark/BigQuery).
- Em ambientes com discos mecânicos ou servidores remotos, o fluxo **XLSX → CSV → Parquet (PyArrow)** pode voltar a ser vantajoso.

---

 *Notebook: 01-Leitura e união de todos os estados.ipynb*  
 *Atualizado em 06/10/2025 – versão 1.0*

Total aproximado: **500 MB**, cobrindo todos os 27 estados brasileiros.

---

## ⚙️ Metodologia de Processamento

### 🔹 Motivação
Foram testados diferentes fluxos de leitura e unificação de planilhas grandes:

| Estratégia | Descrição | Desempenho no Mac M4 |
|-------------|------------|----------------------|
| **XLSX → CSV → Parquet (PyArrow)** | Conversão intermediária para CSV e unificação via PyArrow | Maior tempo total (I/O elevado) |
| **XLSX → XLSX (Pandas nativo)** | Leitura direta de Excel e concatenação incremental | Mais rápido e simples |
| **PyArrow nativo** | Leitura direta via Arrow Dataset | Não suportado oficialmente para `.xlsx` |

🔹 **Conclusão:**  
No ambiente local (Mac M4 com SSD), o fluxo direto **XLSX → XLSX** usando `pandas + openpyxl + ThreadPoolExecutor` apresentou **melhor custo-benefício**, com tempo médio de leitura **~3–5 s por arquivo** e unificação total em **menos de 1 minuto**.

---

## 🧠 Arquitetura de Leitura

A rotina realiza:
1. **Detecção automática** de todos os arquivos `.xlsx` no diretório de trabalho.
2. **Leitura paralela** usando `ThreadPoolExecutor`, otimizando o uso dos núcleos M-series.
3. **Rastreamento da origem** com a coluna `__arquivo_origem`.
4. **Concatenação final** (`pd.concat`) e auditoria de consistência (linhas, colunas, duplicatas).

O código é estruturado para garantir reprodutibilidade, consistência e rastreabilidade.

---

## 🚀 Resultado

- Base nacional consolidada em memória (`base`) com:
  - **>500 mil linhas**
  - **colunas padronizadas**
  - campo `__arquivo_origem` permitindo auditorias
- Tempo total de processamento: **~50 s**
- Exportação final: `base_unificada.xlsx` (ou `.parquet`, conforme necessidade)

---

## 💡 Considerações finais

- O formato `.xlsx` é suficiente nesta etapa, pois os datasets da Anatel têm estrutura tabular estável e colunas homogêneas.
- A conversão para `.parquet` é recomendada apenas **após o pré-processamento**, caso o volume cresça (ex.: >1 M linhas ou uso em Spark/BigQuery).
- Em ambientes com discos mecânicos ou servidores remotos, o fluxo **XLSX → CSV → Parquet (PyArrow)** pode voltar a ser vantajoso.

---

📎 *Notebook: 01-Leitura e união de todos os estados.ipynb*  
🗓️ *Atualizado em 06/10/2025 – versão 1.0*

# 1. Dependências

## Dependências e Configuração do Ambiente

### Visão geral
Esta etapa garante que o ambiente possua as bibliotecas necessárias para leitura paralela e eficiente de múltiplas planilhas `.xlsx` da Anatel.  
O foco é o uso de **pandas** com o **engine openpyxl**, já que o engine `pyarrow` ainda não oferece suporte completo a Excel.

---

### Requisitos principais

| Biblioteca | Função | Versão recomendada | Observações |
|-------------|---------|-------------------|--------------|
| **pandas** | Leitura, concatenação e manipulação de dados | ≥ 2.2.2 | Uso da API moderna e `dtype_backend="pyarrow"` |
| **openpyxl** | Engine de leitura `.xlsx` | ≥ 3.1.2 | Rápido e estável no macOS |
| **pyarrow** | Backend para parquet (opcional) | ≥ 15.0.0 | Utilizado apenas na exportação |
| **tqdm** | Barras de progresso | ≥ 4.66 | Melhora a observabilidade durante a leitura |
| **concurrent.futures** | Execução paralela (nativo do Python) | — | Incluído na biblioteca padrão |

In [1]:
### Instalação (recomendada para ambientes locais ou virtuais)
# !pip install -U "pandas>=2.2.2" "pyarrow>=15" openpyxl tqdm
# !pip install python-calamine

In [10]:
# Configuração inicial no notebook

import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm import tqdm
from python_calamine import CalamineWorkbook
import re

# Mostrar versões para reprodutibilidade
print(f"pandas: {pd.__version__}")
try:
    import pyarrow as pa
    import openpyxl
    print(f"pyarrow: {pa.__version__}")
    print(f"openpyxl: {openpyxl.__version__}")
except ImportError:
    print("⚠️ Algumas dependências opcionais não foram encontradas.")

pandas: 2.3.3
pyarrow: 21.0.0
openpyxl: 3.1.5


### Estratégia de performance
	•	Leitura paralela: utiliza ThreadPoolExecutor com até min(8, os.cpu_count()) workers.
→ Ideal para o Mac M4, que possui múltiplos núcleos de alta eficiência.
	•	Fallback automático: tenta engine="pyarrow" e, se indisponível, muda para engine="openpyxl".
	•	Rastreamento: adiciona a coluna __arquivo_origem para cada planilha lida, permitindo auditoria posterior.
	•	Concatenação segura: pd.concat(..., ignore_index=True) com checagem de duplicatas ao final.

# 2. Estrutura de Diretórios e Detecção Automática de Arquivos


### Objetivo
Padronizar a **descoberta, validação e inventário** dos arquivos `RQUAL_8ind-*.xlsx` na mesma pasta do notebook, antes da leitura paralela.  
Esta etapa evita “surpresas” (arquivos faltando/duplicados, nomes fora do padrão, tamanhos anômalos).

---

### Organização recomendada de pastas:
├─ 01-Leitura e união de todos os estados.ipynb     # este notebook
├─ RQUAL_8ind-A.xlsx
├─ RQUAL_8ind-BA.xlsx
├─ RQUAL_8ind-CD.xlsx
├─ RQUAL_8ind-E-G-MA.xlsx
├─ RQUAL_8ind-MG.xlsx
├─ RQUAL_8ind-MT-MS.xlsx
├─ RQUAL_8ind-P.xlsx
├─ RQUAL_8ind-RJ.xlsx
├─ RQUAL_8ind-RN-RO-RR.xlsx
├─ RQUAL_8ind-RS.xlsx
├─ RQUAL_8ind-SC-SE-TO.xlsx
└─ RQUAL_8ind-SP.xlsx

> 💡**Regra de ouro:** manter todos os `.xlsx` lado a lado com o notebook facilita rastreabilidade e portabilidade.

In [3]:
### 2.1 Descoberta e inventário dos arquivos

PASTA = Path(".")
padrao = "RQUAL_8ind-*.xlsx"

arquivos = sorted(PASTA.glob(padrao))
assert arquivos, f"Nenhum arquivo encontrado com o padrão: {padrao}"

# Inventário (nome, tamanho em MB, data modificação)
inv = pd.DataFrame({
    "arquivo": [a.name for a in arquivos],
    "tamanho_MB": [round(a.stat().st_size / (1024*1024), 2) for a in arquivos],
    "modificado_em": [pd.to_datetime(a.stat().st_mtime, unit="s") for a in arquivos],
}).sort_values("arquivo").reset_index(drop=True)

total_mb = inv["tamanho_MB"].sum()

print(f"📂 {len(inv)} arquivos encontrados ({total_mb:.2f} MB no total)\n")
inv

📂 12 arquivos encontrados (429.15 MB no total)



,arquivo,tamanho_MB,modificado_em
0,RQUAL_8ind-A.xlsx,13.02,2025-04-28 17:19:06
1,RQUAL_8ind-BA.xlsx,32.03,2025-04-28 17:24:36
2,RQUAL_8ind-CD.xlsx,12.25,2025-04-28 17:25:34
3,RQUAL_8ind-E-G-MA.xlsx,42.06,2025-04-28 17:29:14
4,RQUAL_8ind-MG.xlsx,69.30,2025-04-28 18:03:54
5,RQUAL_8ind-MT-MS.xlsx,14.26,2025-04-28 18:13:46
6,RQUAL_8ind-P.xlsx,79.40,2025-04-28 18:17:36
7,RQUAL_8ind-RJ.xlsx,8.57,2025-04-28 18:42:48
8,RQUAL_8ind-RN-RO-RR.xlsx,14.46,2025-04-28 18:43:58
9,RQUAL_8ind-RS.xlsx,42.45,2025-04-28 18:51:34


In [4]:
# 2.2 Validações rápidas de sanidade
#	•	1) Todos os nomes seguem o padrão?
#	•	2) Há duplicatas de nome?
#	•	3) Algum arquivo muito pequeno (< 50 KB) ou muito grande (> 150 MB)? (sinais de exportação incompleta/corrompida)

# 1) Padrão já garantido pelo glob - opcionalmente verificar por regex


regex = re.compile(r"^RQUAL_8ind-.*\.xlsx$", re.IGNORECASE)
nomes_ok = all(regex.match(a) for a in inv["arquivo"])
print("✔️ Padrão de nomes OK?" , nomes_ok)

# 2) Duplicatas de nome
dups = inv[inv["arquivo"].duplicated(keep=False)]
if len(dups):
    print("⚠️ Duplicatas detectadas:\n", dups)
else:
    print("✔️ Sem duplicatas de nome")

# 3) Heurística de tamanhos anômalos
pequenos = inv.loc[inv["tamanho_MB"] < 0.05]     # < 50 KB
grandes  = inv.loc[inv["tamanho_MB"] > 150.00]   # > 150 MB (ajuste se necessário)

if not pequenos.empty:
    print("\n⚠️ Arquivos muito pequenos (possível exportação incompleta):")
    display(pequenos)

if not grandes.empty:
    print("\n⚠️ Arquivos muito grandes (verificar se não há colunas extras/aba errada):")
    display(grandes)

✔️ Padrão de nomes OK? True
✔️ Sem duplicatas de nome


In [5]:
# 2.3 Verificar esquema de colunas antes de ler tudo

# Escolha seu engine principal aqui (openpyxl é estável; 'calamine' é mais rápido se instalado)
ENGINE_PREFERIDO = "openpyxl"  # ou "calamine" se você instalou pandas-calamine

schemas = {}
for a in arquivos:
    cols = list(pd.read_excel(a, engine=ENGINE_PREFERIDO, nrows=0).columns)
    schemas[a.name] = cols

# Interseção & União dos conjuntos de colunas
sets = {k: set(v) for k, v in schemas.items()}
inter = set.intersection(*sets.values())
uni   = set.union(*sets.values())

print(f"✔️ Colunas em comum entre todos: {len(inter)}")
print(f"❗ Colunas totais (união): {len(uni)}")

# Quais arquivos estão com colunas faltando vs. a união?
diffs = {k: sorted(list(uni - v)) for k, v in sets.items() if v != uni}
if diffs:
    print("\n⚠️ Diferenças de colunas por arquivo (faltando em relação à união):")
    for k, d in diffs.items():
        print(f" - {k}: {d[:8]}{'...' if len(d) > 8 else ''}")
else:
    print("✔️ Esquemas perfeitamente alinhados entre todos os arquivos.")

✔️ Colunas em comum entre todos: 18
❗ Colunas totais (união): 18
✔️ Esquemas perfeitamente alinhados entre todos os arquivos.


# 3. Leitura Paralela e Unificação (XLSX → XLSX)

### Objetivo
Ler **todos os arquivos `RQUAL_8ind-*.xlsx`** em paralelo e unificar em um único `DataFrame`, preservando a **rastreabilidade** por meio da coluna `__arquivo_origem`.

> **Engine**: priorizamos `calamine` (se instalado) por ser mais rápido; caso contrário, usamos `openpyxl` (estável e compatível).

In [11]:
# 3.1 Código — leitura paralela + concat

import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm import tqdm

# Configurações
PASTA = Path(".")
PADRAO = "RQUAL_8ind-*.xlsx"
MAX_WORKERS = min(8, (os.cpu_count() or 4))  # bom equilíbrio para M-series

# Detectar arquivos
arquivos = sorted(PASTA.glob(PADRAO))
assert arquivos, f"Nenhum arquivo encontrado com o padrão: {PADRAO}"

# Escolher engine (calamine se disponível; senão openpyxl)
ENGINE = "calamine"
try:
    pd.read_excel(arquivos[0], engine=ENGINE, nrows=0)
except Exception:
    ENGINE = "openpyxl"

print(f"🔧 Engine selecionado: {ENGINE}")
print(f"📂 {len(arquivos)} arquivos serão lidos em paralelo.\n")

def ler_um(arq: Path) -> pd.DataFrame:
    # dtype_backend="pyarrow" reduz memória e melhora concatenação (pandas ≥ 2.2)
    df = pd.read_excel(arq, engine=ENGINE, dtype_backend="pyarrow")
    df["__arquivo_origem"] = arq.name
    return df

# Leitura paralela
dfs = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = {ex.submit(ler_um, a): a for a in arquivos}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="🚀 Lendo planilhas"):
        dfs.append(fut.result())

# Unificação
base = pd.concat(dfs, ignore_index=True)
print(f"\n✅ Base unificada: {len(base):,} linhas × {len(base.columns)} colunas")

🔧 Engine selecionado: calamine
📂 12 arquivos serão lidos em paralelo.



🚀 Lendo planilhas: 100%|███████████████████████| 12/12 [00:52<00:00,  4.38s/it]



✅ Base unificada: 5,962,723 linhas × 19 colunas


In [ ]:
# Tempo de leitura - 00:00:52''; segundos
# Base unificada: 5,962,723 linhas × 19 colunas

In [12]:
# 3.2 Checagens rápidas pós-leitura
# Linhas por arquivo de origem (auditoria)
cont_por_arquivo = base["__arquivo_origem"].value_counts().sort_index()
print("🧾 Linhas por arquivo:")
display(cont_por_arquivo.to_frame("linhas"))

# Duplicatas exatas (opcional): ajusta subset conforme sua chave
# Ex.: subset = colunas que definem unicidade no RQUAL
subset = [c for c in base.columns if c != "__arquivo_origem"]
dup = base.duplicated(subset=subset, keep=False).sum()
print(f"🔁 Possíveis duplicatas (subset={len(subset)} colunas): {dup:,}")

🧾 Linhas por arquivo:


,linhas
__arquivo_origem,
RQUAL_8ind-A.xlsx,205172
RQUAL_8ind-BA.xlsx,424861
RQUAL_8ind-CD.xlsx,195348
RQUAL_8ind-E-G-MA.xlsx,554322
RQUAL_8ind-MG.xlsx,995393
RQUAL_8ind-MT-MS.xlsx,223589
RQUAL_8ind-P.xlsx,1048565
RQUAL_8ind-RJ.xlsx,134497
RQUAL_8ind-RN-RO-RR.xlsx,229476


🔁 Possíveis duplicatas (subset=18 colunas): 0


# 4. Auditoria de Esquema e Consistência

In [13]:
# 4.1 Conferência de colunas comuns e divergentes

# Colunas comuns (presentes em todas) x união (todas as distintas)
# Considerando os 'schemas' coletados na seção 2.3 (se você executou)
try:
    schemas
except NameError:
    # criar schemas rapidamente agora (cabeçalho apenas)
    schemas = {}
    for a in arquivos:
        cols = list(pd.read_excel(a, engine=ENGINE, nrows=0).columns)
        schemas[a.name] = cols

sets = {k: set(v) for k, v in schemas.items()}
inter = set.intersection(*sets.values())
uni   = set.union(*sets.values())

print(f"✔️ Colunas em comum: {len(inter)}")
print(f"❗ Colunas na união: {len(uni)}")

# Se precisar garantir homogeneidade: reordenar/alinhar
if len(uni) != len(inter):
    faltantes = {k: sorted(list(uni - v)) for k, v in sets.items() if v != uni}
    if faltantes:
        print("\n⚠️ Diferenças de colunas por arquivo (faltando em relação à união):")
        for k, d in faltantes.items():
            print(f" - {k}: {d[:10]}{'...' if len(d) > 10 else ''}")

✔️ Colunas em comum: 18
❗ Colunas na união: 18


# 5. Persistência (parquet)

In [15]:
# Parquet (compacto e muito rápido de abrir)
base.to_parquet("base_RQUAL_unificada.parquet", index=False)
print("📦 Salvo Parquet: base_RQUAL_unificada.parquet")

📦 Salvo Parquet: base_RQUAL_unificada.parquet


In [18]:
# ======================================================
# 6️⃣ Validação Final da Base Unificada (formato Parquet)
# ======================================================
import pandas as pd
from pathlib import Path

# Caminho do arquivo final
arquivo_final = Path("base_unificada.parquet")

# Leitura com engine PyArrow (super eficiente)
df = pd.read_parquet(arquivo_final, engine="pyarrow")

print(f"✅ Arquivo carregado: {arquivo_final.name}")
print(f"→ Linhas: {len(df):,} | Colunas: {len(df.columns)}\n")

# Amostra das primeiras linhas
print("🔹 Colunas iniciais:")
display(df.head(3))

# Estatísticas gerais — numéricas
print("\n📊 Estatísticas descritivas — numéricas")
display(df.describe())

# Estatísticas gerais — categóricas
print("\n🔠 Estatísticas descritivas — categóricas")
display(df.describe(include=['object', 'string']))

# Verificação rápida de colunas nulas ou vazias
print("\n🧩 Colunas com valores ausentes:")
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
display(faltantes)

# Identificação de colunas totalmente vazias (se houver)
vazias = [c for c in df.columns if df[c].isna().all()]
if vazias:
    print(f"\n⚠️ Colunas 100% vazias: {len(vazias)}")
    print(vazias)
else:
    print("\n✅ Nenhuma coluna completamente vazia encontrada.")

✅ Arquivo carregado: base_unificada.parquet
→ Linhas: 5,962,723 | Colunas: 19

🔹 Colunas iniciais:


,Ano,Mês,Serviço,Tipo,Grupo,UF,Município,Código IBGE,Prestadora,Indicador,Descrição,Resultado,Unidade de Medida,Valor de Referência Superior,Valor de Referência Inferior,Erro Amostral,Limite Inferior,Limite Superior,__arquivo_origem
0,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AL,Maceió,2704302,ALGAR,IND5,Latência bidirecional da conexão de dados,0,%,0.9500000000000002,0.65,70.7,0,70.7083,RQUAL_8ind-A.xlsx
1,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AP,Macapá,1600303,CLARO,IND5,Latência bidirecional da conexão de dados,100,%,0.9500000000000002,0.65,31.62,68.3783,100,RQUAL_8ind-A.xlsx
2,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AL,Maceió,2704302,CLARO,IND5,Latência bidirecional da conexão de dados,78.7878,%,0.9500000000000002,0.65,10.65,68.1282,89.4474,RQUAL_8ind-A.xlsx



📊 Estatísticas descritivas — numéricas


,Ano,Mês,Código IBGE
count,5.962723e+06,5.962723e+06,5.962723e+06
mean,2.023234e+03,6.573596e+00,3.311105e+06
std,8.998509e-01,3.450891e+00,9.506536e+05
min,2.022000e+03,1.000000e+00,1.100015e+06
25%,2.023000e+03,3.000000e+00,2.615003e+06
50%,2.023000e+03,7.000000e+00,3.163300e+06
75%,2.024000e+03,1.000000e+01,4.122701e+06
max,2.025000e+03,1.200000e+01,5.300108e+06



🔠 Estatísticas descritivas — categóricas


,Serviço,Tipo,Grupo,UF,Município,Prestadora,Indicador,Descrição,Resultado,Unidade de Medida,Valor de Referência Superior,Valor de Referência Inferior,Erro Amostral,Limite Inferior,Limite Superior,__arquivo_origem
count,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723,5962723
unique,4,2,2,27,5297,12,8,8,481729,3,7,7,2974,321345,304543,12
top,Telefonia Móvel,Indicador IQS,Redes,MG,Santa Luzia,OI,IND8,Disponibilidade,NO,%,-,-,-,-,-,RQUAL_8ind-P.xlsx
freq,1977042,4684939,3556838,995393,5286,1828112,1621769,1621769,1492130,4684939,2407861,2407861,4692772,4692772,4692772,1048565



🧩 Colunas com valores ausentes:


Series([], dtype: int64)


✅ Nenhuma coluna completamente vazia encontrada.
